# Partie 2 — Covariables Environnementales & Étude d'Ablation

**Objectif** : évaluer l'apport du Climat, Sol et Topographie à la classification des cultures par MCTNet via une _early fusion_ sur 5 configurations.

| # | Configuration | Données d'entrée | `input_dim` |
|---|---------------|-----------------|-------------|
| 1 | `baseline`    | S2 seul           | 10 |
| 2 | `climate`     | S2 + Climat (3)   | 13 |
| 3 | `soil`        | S2 + Sol (3)      | 13 |
| 4 | `topography`  | S2 + Topo (2)     | 12 |
| 5 | `all`         | S2 + tout (8)     | 18 |

**Stratégie de fusion** : les covariables statiques `[E]` sont répétées sur
les 36 pas de temps → `[36, E]`, puis concaténées aux bandes Sentinel-2 `[36, 10]`
avant l'entrée dans MCTNet. L'architecture interne n'est pas modifiée.

---
## Cellule 1 — Montage Drive & dépendances

In [ ]:
import subprocess, sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                        'numpy', 'pandas', 'matplotlib', 'scikit-learn'])

import json
from pathlib import Path
from types  import SimpleNamespace
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

print(f'PyTorch {torch.__version__} | GPU : {torch.cuda.is_available()}')

---
## Cellule 2 — Configuration centralisée

In [ ]:
# ── Chemins ──────────────────────────────────────────────────────────────────
PROJECT_DIR = Path('/content/drive/MyDrive/New project/python')
BASE_DIR    = Path('/content/drive/MyDrive/mctnet_env_covariates_2021')

# CSV exportés par mctnet_env_covariates_prep_2021.js
CSV_AR = BASE_DIR / 'mctnet_env_samples_AR_2021.csv'
CSV_CA = BASE_DIR / 'mctnet_env_samples_CA_2021.csv'

PROCESSED_ENV_DIR = BASE_DIR / 'processed_env'
EDA_DIR           = BASE_DIR / 'eda_results'
ABLATION_DIR      = BASE_DIR / 'ablation_runs'

for d in [PROCESSED_ENV_DIR, EDA_DIR, ABLATION_DIR]:
    d.mkdir(parents=True, exist_ok=True)

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

# ── Hyperparamètres d'ablation ────────────────────────────────────────────────
# Réduire epochs à 50 pour l'ablation (100 pour le rapport final)
CFG = SimpleNamespace(
    epochs                  = 50,
    batch_size              = 32,
    learning_rate           = 1e-3,
    weight_decay            = 1e-4,
    dropout                 = 0.1,
    n_stages                = 3,
    n_heads                 = 5,
    kernel_size             = 3,
    seed                    = 2021,
    num_workers             = 0,
    early_stopping_patience = 10,
    cpu                     = False,
    no_alpe = False, no_mask = False, no_cnn = False, no_trans = False,
    # États et configurations à exécuter
    states  = ['arkansas', 'california'],
    configs = ['baseline', 'climate', 'soil', 'topography', 'all'],
)

print(f'CSV AR : {CSV_AR} (existe = {CSV_AR.exists()})')
print(f'CSV CA : {CSV_CA} (existe = {CSV_CA.exists()})')

---
## Cellule 3 — Vérification de cohérence GEE ↔ Python

Vérifie que les noms de colonnes du script JavaScript correspondent exactement
aux listes Python dans `build_mctnet_env_dataset.py`.

In [ ]:
from build_mctnet_env_dataset import (
    CLIMATE_COLUMNS, SOIL_COLUMNS, TOPOGRAPHY_COLUMNS,
    ALL_ENV_COLUMNS, ENV_GROUPS, ABLATION_CONFIGS
)

# Sous-ensemble sélectionné parmi les colonnes exportées par GEE
# (le script JS exporte 14 vars ; on en retient 8 pour ce projet)
GEE_CLIMATE_BANDS = [
    'climate_vpd_mean_kpa', 'climate_pr_sum_mm', 'climate_tmmn_mean_c',
]
GEE_SOIL_BANDS = [
    'soil_clay_0cm_pct', 'soil_sand_0cm_pct', 'soil_phh2o_0cm',
]
GEE_TOPO_BANDS = [
    'topo_elevation_m', 'topo_slope_deg',
]

checks = [
    ('Climat',      GEE_CLIMATE_BANDS, CLIMATE_COLUMNS),
    ('Sol',         GEE_SOIL_BANDS,    SOIL_COLUMNS),
    ('Topographie', GEE_TOPO_BANDS,    TOPOGRAPHY_COLUMNS),
]

all_ok = True
for group, gee_cols, py_cols in checks:
    # Tri identique pour la comparaison (l'ordre GEE peut différer)
    ok = sorted(gee_cols) == sorted(py_cols)
    status = '✓ OK' if ok else '✗ DIFFÉRENT'
    print(f'{group:12s} ({len(py_cols)} vars retenues) : {status}')
    if not ok:
        all_ok = False
        diff = set(gee_cols) ^ set(py_cols)
        print(f'  Différences : {diff}')

print(f'\nTotal colonnes env retenues : {len(ALL_ENV_COLUMNS)}  (3 clim + 3 sol + 2 topo)')
print(f'Configs ablation : {list(ABLATION_CONFIGS.keys())}')
print(f'input_dim par config : baseline=10, climate=13, soil=13, topography=12, all=18')
print(f'\n{"✓ Cohérence GEE / Python confirmée." if all_ok else "✗ Corriger les noms de colonnes."}')

---
## Cellule 4 — Préparation des datasets avec covariables

In [ ]:
from build_mctnet_env_dataset import build_env_dataset_bundle, save_bundle
from build_dataset import detect_state

for csv_path in [CSV_AR, CSV_CA]:
    if not csv_path.exists():
        raise FileNotFoundError(
            f'CSV manquant : {csv_path}\n'
            f'Exportez-le depuis GEE via mctnet_env_covariates_prep_2021.js'
        )

    bundle, meta = build_env_dataset_bundle(
        csv_path              = csv_path,
        normalize_reflectance = True,
        reflectance_scale     = 10000.0,
        split_seed            = CFG.seed,
    )

    slug      = detect_state(pd.DataFrame({'state_name': [meta['state_name']]}))
    npz_path  = PROCESSED_ENV_DIR / f'{slug}_mctnet_env_dataset.npz'
    json_path = PROCESSED_ENV_DIR / f'{slug}_mctnet_env_dataset.json'
    save_bundle(bundle, meta, npz_path, json_path)

    env_shape = meta['environmental_covariates']['n_env_features']
    print(f'\n[{meta["state_name"]}]')
    print(f'  Classes ({meta["num_classes"]}) : {list(meta["class_name_to_index"].keys())}')
    print(f'  x shape    : [N, 36, 10]')
    print(f'  env shape  : [N, {env_shape}]   ({len(CLIMATE_COLUMNS)} clim + {len(SOIL_COLUMNS)} sol + {len(TOPOGRAPHY_COLUMNS)} topo)')
    for split, counts in meta['split_counts'].items():
        print(f'  {split:5s} ({meta["samples_per_split"][split]:5d}) : {counts}')

print('\nDatasets env prêts.')

---
## Cellule 5 — Analyse Exploratoire (EDA)

Génère :
- Matrice de corrélation de Pearson entre covariables
- Heatmap des moyennes par classe de culture
- Barplot du ratio η (Eta) corrélation covariable ↔ classe

In [ ]:
from environmental_covariate_eda import run_eda

eda_artifacts = {}
for csv_path, state_name in [(CSV_AR, 'Arkansas'), (CSV_CA, 'California')]:
    if not csv_path.exists(): continue
    state_eda_dir = EDA_DIR / state_name
    print(f'EDA → {state_name}...')
    eda_artifacts[state_name] = run_eda(csv_path=csv_path, output_dir=state_eda_dir)
    print(f'  OK : {list(eda_artifacts[state_name].keys())}')

print('\nEDA terminée.')

### Visualisation EDA

In [ ]:
from IPython.display import display, Image as IPImage

for state_name in ['Arkansas', 'California']:
    arts = eda_artifacts.get(state_name, {})
    if not arts: continue

    print(f'\n══ EDA : {state_name} ══')

    print('Matrice de corrélation de Pearson entre covariables :')
    display(IPImage(filename=arts['pearson_heatmap_png']))

    print('Ratio η — corrélation covariable ↔ classe de culture :')
    display(IPImage(filename=arts['eta_barplot_png']))

    print('Moyennes des covariables par classe :')
    display(IPImage(filename=arts['class_means_heatmap_png']))

    # Résumé textuel de l'EDA
    eta_df = pd.read_csv(arts['eta_csv'], index_col=0)
    print('\nTop 5 covariables les plus discriminantes (Eta) :')
    print(eta_df.sort_values('eta', ascending=False).head(5).to_string())

---
## Cellule 6 — Étude d'Ablation (5 configurations)

Lance `run_ablation_study.run_ablation_experiment()` pour chaque état.
Sauvegarde dans `ABLATION_DIR/{state}/{config}/` :
- checkpoint `.pt`
- historique d'entraînement `.json`
- matrice de confusion `.png` (meilleure config)

In [ ]:
from run_ablation_study import run_ablation_experiment, set_seed

set_seed(CFG.seed)

ablation_results = run_ablation_experiment(
    processed_env_dir = PROCESSED_ENV_DIR,
    output_dir        = ABLATION_DIR,
    states            = CFG.states,
    args              = CFG,
)

print('\nAblation terminée.')

---
## Cellule 7 — Visualisation des résultats d'ablation

In [ ]:
# Barplots OA / F1 / Kappa
from run_ablation_study import plot_ablation_barplot, plot_all_metrics_grid

for metric in ['oa', 'macro_f1', 'kappa']:
    png = ABLATION_DIR / f'ablation_{metric}_barplot.png'
    if png.exists():
        display(IPImage(filename=str(png)))

grid_png = ABLATION_DIR / 'ablation_metrics_grid.png'
if grid_png.exists():
    print('\nGrille récapitulative :')
    display(IPImage(filename=str(grid_png)))

---
## Cellule 8 — Tableau récapitulatif final

In [ ]:
summary_csv = ABLATION_DIR / 'ablation_summary.csv'
if summary_csv.exists():
    df = pd.read_csv(summary_csv)

    # Pivot pour une lecture facile état × config
    pivot_oa = df.pivot(index='config', columns='state', values='oa').round(4)
    pivot_f1 = df.pivot(index='config', columns='state', values='macro_f1').round(4)

    print('Overall Accuracy (OA) :')
    print(pivot_oa.to_string())

    print('\nMacro F1 :')
    print(pivot_f1.to_string())

    # Amélioration vs baseline
    print('\nAmélioration OA vs Baseline :')
    for state in CFG.states:
        baseline_oa = df[(df.state == state) & (df.config == 'baseline')]['oa'].values
        if len(baseline_oa) == 0: continue
        baseline_oa = baseline_oa[0]
        state_rows = df[df.state == state].copy()
        state_rows['delta_oa'] = (state_rows['oa'] - baseline_oa).round(4)
        print(f'  {state.title()}:')
        print(state_rows[['config', 'oa', 'macro_f1', 'kappa', 'delta_oa']].to_string(index=False))

---
## Cellule 9 — Matrices de confusion des meilleures configurations

In [ ]:
for state in CFG.states:
    state_dir = ABLATION_DIR / state
    for png in sorted(state_dir.glob('confusion_matrix_best_*.png')):
        print(f'\n── {state.title()} : {png.stem} ──')
        display(IPImage(filename=str(png)))

---
## Annexe — Interprétation EDA

Le ratio η (eta) mesure la corrélation entre une variable continue et une variable
catégorielle (classe de culture). Un η élevé indique que la covariable
**discrimine bien les classes**.

| Groupe | Variables retenues | Intérêt discriminant |
|--------|-------------------|----------------------|
| Climat | `climate_vpd_mean_kpa`, `climate_pr_sum_mm`, `climate_tmmn_mean_c` | Régime hydrique et thermique lié aux cycles phénologiques |
| Sol    | `soil_clay_0cm_pct`, `soil_sand_0cm_pct`, `soil_phh2o_0cm` | Texture et pH conditionnent les cultures en sec vs irriguées |
| Topo   | `topo_elevation_m`, `topo_slope_deg` | L'altitude et la pente contraignent les types de cultures |

**Interprétation attendue du barplot ablation** :
- Si `climate > baseline` : VPD et précipitations aident à séparer les cultures à calendrier hydrique distinct (riz irrigué vs. cultures sèches)
- Si `soil > baseline` : la texture et le pH du sol discriminent les cultures en sec (coton, pistache) des cultures inondées (riz)
- Si `all ≈ best_single` : un groupe domine ; les 8 variables combinées n'apportent pas de gain supplémentaire significatif
